# 🛒 Retail Customer Segmentation & RFM Analysis
**Dataset:** UCI Online Retail (2010–2011)  
**Goal:** Segment customers using Recency, Frequency, and Monetary (RFM) analysis + K-Means Clustering

---
### 📌 Project Steps:
1. Import Libraries
2. Load & Explore Data
3. Data Cleaning & Preprocessing
4. RFM Metric Calculation
5. RFM Scoring
6. Customer Segmentation (Rule-based)
7. K-Means Clustering
8. Visualization & Insights

---
## 📦 Step 1: Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Date handling
from datetime import datetime, timedelta

# Settings
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✅ All libraries imported successfully!')

---
## 📂 Step 2: Load & Explore Data

In [ ]:
# -------------------------------------------------------
# Load the dataset
# NOTE: Change the filename/path if needed
# -------------------------------------------------------
df = pd.read_excel('Online Retail.xlsx', engine='openpyxl')  
# If you have a .csv file, use:
# df = pd.read_csv('Online Retail.csv', encoding='unicode_escape')

print(f'📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'📅 Date Range: {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
df.head(10)

In [ ]:
# Basic info
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('🔍 Missing Values:')
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Check duplicate rows
dupes = df.duplicated().sum()
print(f'🔁 Duplicate rows: {dupes:,}')

# Unique countries
print(f'🌍 Countries: {df["Country"].nunique()}')
print(df['Country'].value_counts().head(10))

---
## 🧹 Step 3: Data Cleaning & Preprocessing

In [ ]:
df_clean = df.copy()

# 1. Remove rows with missing CustomerID (can't do RFM without it)
df_clean.dropna(subset=['CustomerID'], inplace=True)
print(f'✅ After dropping missing CustomerID: {df_clean.shape[0]:,} rows')

# 2. Remove duplicate rows
df_clean.drop_duplicates(inplace=True)
print(f'✅ After removing duplicates: {df_clean.shape[0]:,} rows')

# 3. Remove cancelled orders (InvoiceNo starting with 'C')
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
print(f'✅ After removing cancellations: {df_clean.shape[0]:,} rows')

# 4. Remove negative or zero Quantity and UnitPrice
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
print(f'✅ After removing invalid Quantity/Price: {df_clean.shape[0]:,} rows')

# 5. Fix data types
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# 6. Create TotalPrice column
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(f'\n🎯 Final clean dataset: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')
df_clean.head()

---
## 📐 Step 4: RFM Metric Calculation

| Metric | Definition |
|--------|------------|
| **Recency** | How many days ago was the last purchase? (Lower = Better) |
| **Frequency** | How many times did the customer purchase? (Higher = Better) |
| **Monetary** | How much total money did the customer spend? (Higher = Better) |

In [ ]:
# Reference date = 1 day after the last transaction in the dataset
reference_date = df_clean['InvoiceDate'].max() + timedelta(days=1)
print(f'📅 Reference Date (Snapshot Date): {reference_date.date()}')

# Calculate RFM for each customer
rfm = df_clean.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate',  lambda x: (reference_date - x.max()).days),
    Frequency = ('InvoiceNo',    'nunique'),
    Monetary  = ('TotalPrice',   'sum')
).reset_index()

print(f'\n👥 Total Unique Customers: {rfm.shape[0]:,}')
rfm.head(10)

In [ ]:
# RFM Summary Statistics
print('📊 RFM Summary Statistics:')
rfm[['Recency','Frequency','Monetary']].describe()

In [ ]:
# Visualize RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('RFM Distribution', fontsize=16, fontweight='bold')

axes[0].hist(rfm['Recency'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Recency (days since last purchase)')
axes[0].set_xlabel('Days')

axes[1].hist(rfm['Frequency'], bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Frequency (number of purchases)')
axes[1].set_xlabel('Count')

axes[2].hist(rfm['Monetary'], bins=30, color='seagreen', edgecolor='white')
axes[2].set_title('Monetary (total spend in £)')
axes[2].set_xlabel('£ Amount')

plt.tight_layout()
plt.savefig('rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🎯 Step 5: RFM Scoring (1–5 Scale)

We divide each metric into **5 equal quantiles (quintiles)**:
- **Recency**: Lower days = Score 5 (best), Higher days = Score 1 (worst)
- **Frequency**: Higher count = Score 5 (best)
- **Monetary**: Higher spend = Score 5 (best)

In [ ]:
# Score Recency: REVERSED (lower recency = higher score)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# Score Frequency: higher = better
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Score Monetary: higher = better
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Combined RFM Score (string for segmentation)
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Total numeric RFM score
rfm['RFM_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print('✅ RFM Scores assigned!')
rfm.head(10)

---
## 🏷️ Step 6: Customer Segmentation (Rule-Based)

In [ ]:
# Segmentation map based on R and F scores
def segment_customer(row):
    r = row['R_Score']
    f = row['F_Score']
    m = row['M_Score']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f <= 2:
        return 'Potential Loyalists'
    elif r == 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f >= 4:
        return 'Cannot Lose Them'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost Customers'
    elif r == 2 and f == 2:
        return 'Hibernating'
    else:
        return 'About To Sleep'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# Segment distribution
segment_counts = rfm['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Count']
segment_counts['Percentage'] = (segment_counts['Count'] / segment_counts['Count'].sum() * 100).round(2)
print('📊 Customer Segments:')
segment_counts

In [ ]:
# Visualize Segments — Pie Chart
colors = ['#2ecc71','#3498db','#9b59b6','#e74c3c','#f39c12','#1abc9c','#e67e22','#e91e63','#00bcd4']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Customer Segmentation', fontsize=16, fontweight='bold')

# Pie Chart
axes[0].pie(segment_counts['Count'], labels=segment_counts['Segment'],
            autopct='%1.1f%%', colors=colors, startangle=140,
            textprops={'fontsize': 10})
axes[0].set_title('Segment Distribution (%)')

# Bar Chart
bars = axes[1].barh(segment_counts['Segment'], segment_counts['Count'],
                    color=colors[:len(segment_counts)])
axes[1].set_xlabel('Number of Customers')
axes[1].set_title('Customers per Segment')
for bar, val in zip(bars, segment_counts['Count']):
    axes[1].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('segment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Segment-wise RFM averages
segment_summary = rfm.groupby('Segment').agg(
    Customers  = ('CustomerID', 'count'),
    Avg_Recency   = ('Recency',   'mean'),
    Avg_Frequency = ('Frequency', 'mean'),
    Avg_Monetary  = ('Monetary',  'mean')
).round(2).reset_index()

segment_summary = segment_summary.sort_values('Avg_Monetary', ascending=False)
print('📋 Segment-wise RFM Summary:')
segment_summary

In [ ]:
# Heatmap of RFM scores per Segment
heatmap_data = rfm.groupby('Segment')[['R_Score','F_Score','M_Score']].mean().round(2)

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlGn',
            linewidths=0.5, cbar_kws={'label': 'Average Score (1–5)'})
plt.title('RFM Score Heatmap by Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rfm_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🤖 Step 7: K-Means Clustering
We use machine learning to find data-driven clusters (not just rule-based).

In [ ]:
# Use log transformation to reduce skewness before clustering
rfm_log = rfm[['Recency','Frequency','Monetary']].copy()
rfm_log['Recency']   = np.log1p(rfm_log['Recency'])
rfm_log['Frequency'] = np.log1p(rfm_log['Frequency'])
rfm_log['Monetary']  = np.log1p(rfm_log['Monetary'])

# Standardize
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

print('✅ Data normalized and scaled for clustering')

In [ ]:
# Elbow Method — find optimal number of clusters (K)
inertia = []
silhouette = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(rfm_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Elbow curve
axes[0].plot(K_range, inertia, 'bo-', markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method — Optimal K')

# Silhouette scores
axes[1].plot(K_range, silhouette, 'rs-', markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score per K')

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = K_range[silhouette.index(max(silhouette))]
print(f'\n🏆 Best K by Silhouette Score: {best_k}')

In [ ]:
# Apply K-Means with best K (adjust if needed)
OPTIMAL_K = 4  # Change based on elbow/silhouette result

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

print(f'✅ K-Means applied with K={OPTIMAL_K}')
print(rfm['Cluster'].value_counts().sort_index())

In [ ]:
# Cluster profile summary
cluster_summary = rfm.groupby('Cluster').agg(
    Customers  = ('CustomerID', 'count'),
    Avg_Recency   = ('Recency',   'mean'),
    Avg_Frequency = ('Frequency', 'mean'),
    Avg_Monetary  = ('Monetary',  'mean')
).round(2)

print('📊 Cluster Profiles:')
cluster_summary

In [ ]:
# 3D Scatter Plot of Clusters
fig = px.scatter_3d(
    rfm, x='Recency', y='Frequency', z='Monetary',
    color='Cluster', color_continuous_scale='Viridis',
    title='3D Customer Clusters (Recency vs Frequency vs Monetary)',
    opacity=0.6,
    labels={'Cluster': 'Cluster'},
    hover_data=['CustomerID', 'Segment']
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# Snake Plot — cluster comparison
rfm_melt = rfm[['Cluster','R_Score','F_Score','M_Score']].copy()
rfm_melt = rfm_melt.groupby('Cluster')[['R_Score','F_Score','M_Score']].mean().reset_index()
rfm_melt = rfm_melt.melt(id_vars='Cluster', var_name='Metric', value_name='Score')

plt.figure(figsize=(10, 5))
sns.lineplot(data=rfm_melt, x='Metric', y='Score', hue='Cluster',
             marker='o', linewidth=2.5, palette='Set1')
plt.title('Snake Plot — Average RFM Score per Cluster', fontsize=14, fontweight='bold')
plt.xlabel('RFM Metric')
plt.ylabel('Average Score (1-5)')
plt.xticks(['R_Score','F_Score','M_Score'], ['Recency', 'Frequency', 'Monetary'])
plt.legend(title='Cluster')
plt.tight_layout()
plt.savefig('snake_plot.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📊 Step 8: Final Insights & Export

In [ ]:
# Top 10 revenue-generating customers
print('💰 Top 10 Customers by Revenue:')
rfm.nlargest(10, 'Monetary')[['CustomerID','Recency','Frequency','Monetary','Segment','Cluster']]

In [ ]:
# Monthly Revenue Trend
df_clean['Month'] = df_clean['InvoiceDate'].dt.to_period('M')
monthly_rev = df_clean.groupby('Month')['TotalPrice'].sum().reset_index()
monthly_rev['Month'] = monthly_rev['Month'].astype(str)

plt.figure(figsize=(14, 5))
plt.plot(monthly_rev['Month'], monthly_rev['TotalPrice'], marker='o', color='steelblue', linewidth=2)
plt.fill_between(monthly_rev['Month'], monthly_rev['TotalPrice'], alpha=0.2, color='steelblue')
plt.title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Revenue by Country (Top 10)
country_rev = df_clean.groupby('Country')['TotalPrice'].sum().nlargest(10).reset_index()

plt.figure(figsize=(12, 5))
sns.barplot(data=country_rev, x='Country', y='TotalPrice', palette='Blues_d')
plt.title('Top 10 Countries by Revenue', fontsize=14, fontweight='bold')
plt.xlabel('Country')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('revenue_by_country.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Export final RFM table
rfm.to_csv('rfm_results.csv', index=False)
print('✅ RFM results saved to rfm_results.csv')
print(f'\n📋 Final Dataset Shape: {rfm.shape}')
rfm.head()

---
## ✅ Project Summary

| Step | What We Did |
|------|-------------|
| Data Loading | Loaded UCI Online Retail dataset (541k+ rows) |
| Cleaning | Removed nulls, duplicates, cancellations, invalid prices |
| RFM Calculation | Computed Recency, Frequency, Monetary per customer |
| RFM Scoring | Scored each metric 1–5 using quantiles |
| Rule-based Segmentation | 9 segments: Champions, Loyal, At Risk, Lost, etc. |
| K-Means Clustering | Data-driven clusters using Elbow + Silhouette method |
| Visualization | Pie chart, heatmap, 3D scatter, snake plot, trends |
| Export | Saved final RFM results to CSV |

### 💡 Business Recommendations:
- **Champions** → Reward them, ask for reviews, upsell premium products
- **Loyal Customers** → Offer loyalty programs, early access to new products
- **At Risk** → Send win-back campaigns, personalized discounts
- **Lost Customers** → Last-chance email offers or let go
- **New Customers** → Onboarding emails, first-purchase discounts